# SafeWatch — YOLOv8n 파인튜닝국내 도로 데이터로 YOLOv8n을 파인튜닝한다.**세션이 끊겼을 때**: 셀 1 → 2 → 3을 다시 실행한 뒤 맨 아래 **재개** 셀만 실행하면 중단 지점부터 이어서 학습한다.

## 1. 환경 확인 및 드라이브 마운트

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics
import ultralytics
print('ultralytics', ultralytics.__version__)

## 2. 설정경로와 하이퍼파라미터를 여기서만 고친다.

In [ ]:
from pathlib import Path

# ── 경로 (본인 환경에 맞게 수정) ───────────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive/safewatch')
ZIP_PATH    = DRIVE_ROOT / 'dataset.zip'      # 업로드한 zip
RUNS_DIR    = DRIVE_ROOT / 'runs'             # 체크포인트 저장 위치 (드라이브)

LOCAL_DATA  = Path('/content/dataset')        # 압축 해제 위치 (로컬 디스크)
DATA_YAML   = LOCAL_DATA / 'data.yaml'        # 해제 후 data.yaml 실제 경로

# ── 학습 설정 ─────────────────────────────────────────────
RUN_NAME    = 'finetune_v1'   # 재실행 시 이름 바꾸지 말 것 (resume가 이 이름을 찾음)
MODEL       = 'yolov8n.pt'    # n 고정 — s 이상은 Pi5에서 실시간 불가
IMGSZ       = 320             # 배포 해상도와 일치시킴
EPOCHS      = 100
BATCH       = 32              # OOM 나면 16으로
PATIENCE    = 20              # 개선 없을 때 조기 종료
SAVE_PERIOD = 5               # N 에폭마다 체크포인트 저장
WORKERS     = 2               # 코랩 vCPU 수 고려

RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('설정 완료')

## 3. 데이터 준비드라이브의 zip을 로컬 디스크로 복사한 뒤 압축을 푼다.드라이브 경로를 학습에 직접 쓰면 파일 입출력 지연으로 크게 느려지므로 반드시 로컬로 옮긴다.

In [ ]:
import os, shutil, subprocess, time

assert ZIP_PATH.exists(), f'zip을 찾을 수 없음: {ZIP_PATH}'
print(f'zip 크기: {os.path.getsize(ZIP_PATH) / 1e9:.2f} GB')
print('로컬 업로드 전 원본 크기와 바이트 단위로 일치하는지 확인할 것')

t0 = time.time()
local_zip = Path('/content/dataset.zip')
if not local_zip.exists():
    shutil.copy(ZIP_PATH, local_zip)
print(f'복사 완료 ({time.time()-t0:.0f}s)')

t0 = time.time()
if not LOCAL_DATA.exists():
    subprocess.run(['unzip', '-q', str(local_zip), '-d', str(LOCAL_DATA)], check=True)
print(f'압축 해제 완료 ({time.time()-t0:.0f}s)')

!ls -la {LOCAL_DATA}

## 4. 사전 점검**본 학습 전에 반드시 통과시킬 것.** 설정 오류를 몇 초 만에 잡아낸다.여기서 걸러내지 못하면 장시간 학습이 중간에 멈추거나, 잘못된 데이터로 끝까지 돌아간다.

In [ ]:
import yaml, glob

# data.yaml 존재 및 내용 확인
assert DATA_YAML.exists(), f'data.yaml 없음: {DATA_YAML}  → LOCAL_DATA 하위 실제 경로 확인'
cfg = yaml.safe_load(open(DATA_YAML))
print('── data.yaml ──')
print(yaml.dump(cfg, allow_unicode=True))

# train/val 경로 해석
base = Path(cfg.get('path', DATA_YAML.parent))
if not base.is_absolute():
    base = DATA_YAML.parent / base

problems = []
for split in ['train', 'val']:
    if split not in cfg:
        problems.append(f'{split} 항목이 data.yaml에 없음')
        continue
    p = Path(cfg[split])
    img_dir = p if p.is_absolute() else base / p
    if not img_dir.exists():
        problems.append(f'{split} 경로 없음: {img_dir}')
        continue

    imgs = [f for ext in ('*.jpg','*.jpeg','*.png')
            for f in glob.glob(str(img_dir / '**' / ext), recursive=True)]
    lbl_dir = Path(str(img_dir).replace('/images', '/labels'))
    lbls = glob.glob(str(lbl_dir / '**' / '*.txt'), recursive=True)

    print(f'{split}: 이미지 {len(imgs)}장 / 라벨 {len(lbls)}개  ({img_dir})')
    if len(imgs) == 0:
        problems.append(f'{split} 이미지가 0장')
    if len(lbls) == 0:
        problems.append(f'{split} 라벨이 0개 — labels 폴더 구조 확인 필요')
    elif abs(len(imgs) - len(lbls)) > len(imgs) * 0.05:
        problems.append(f'{split} 이미지/라벨 개수 차이 큼 ({len(imgs)} vs {len(lbls)})')

print()
if problems:
    for p in problems:
        print('[문제]', p)
    raise SystemExit('위 문제를 해결한 뒤 진행할 것')
print('사전 점검 통과')

라벨 파일 내용을 눈으로 한 번 확인한다. 클래스 인덱스가 `data.yaml`의 `names` 범위를 벗어나면 학습이 중간에 실패한다.

In [ ]:
import random

sample = random.choice(lbls)
print('샘플 라벨:', sample)
print(open(sample).read()[:400])

nc = len(cfg['names']) if isinstance(cfg['names'], (list, dict)) else cfg.get('nc')
print(f'\n정의된 클래스 수: {nc}')
print('각 줄의 첫 숫자(클래스 인덱스)가 0 이상', nc - 1, '이하인지 확인')

## 5. 본 학습체크포인트는 드라이브에 저장되므로 세션이 끊겨도 남는다.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    save_period=SAVE_PERIOD,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    plots=True,
)
print('학습 완료')

## 6. 결과 확인 및 백업

In [ ]:
run_dir = RUNS_DIR / RUN_NAME
print(run_dir)
!ls -la {run_dir}/weights

# 검증 지표
best = YOLO(str(run_dir / 'weights' / 'best.pt'))
metrics = best.val(data=str(DATA_YAML), imgsz=IMGSZ)
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall   : {metrics.box.mr:.4f}')

파인튜닝 전후 비교를 위해 **사전학습 모델의 동일 데이터 성능**도 측정해 둔다.보고서의 도메인 갭 측정 근거가 되는 수치다.

In [ ]:
baseline = YOLO(MODEL)
m0 = baseline.val(data=str(DATA_YAML), imgsz=IMGSZ)

print('구분        mAP50    mAP50-95  Precision  Recall')
print(f'사전학습    {m0.box.map50:.4f}   {m0.box.map:.4f}    {m0.box.mp:.4f}     {m0.box.mr:.4f}')
print(f'파인튜닝    {metrics.box.map50:.4f}   {metrics.box.map:.4f}    {metrics.box.mp:.4f}     {metrics.box.mr:.4f}')

---## 재개 (세션이 끊긴 경우에만)셀 1 → 2 → 3을 먼저 실행한 뒤 아래를 실행한다. 중단된 에폭부터 이어서 학습한다.

In [ ]:
from ultralytics import YOLO

last = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
assert last.exists(), f'체크포인트 없음: {last}'

model = YOLO(str(last))
model.train(resume=True)